# EX_00 — PyTorch vs TensorFlow (ejercicios)

**Notebook de referencia:** `notebook/00_Frameworks_Pytorch_vs_Tensorflow.ipynb`

**Tiempo orientativo:** ~30 minutos.

En esta hoja practicarás ideas equivalentes en ambos frameworks: tensores, capas lineales y un forward pass mínimo.


## Actividad 1 — Activación a mano

Implementa en NumPy una función `relu` y otra `sigmoid` y comprueba que coinciden con `torch` y `tensorflow` en un vector de prueba.

*Hint:* use `torch.relu`, `tf.nn.relu`; for sigmoid use `torch.sigmoid` and `tf.nn.sigmoid`.


In [1]:
import numpy as np
import torch
import tensorflow as tf

# TODO: implement relu_np(z) and sigmoid_np(z)

def relu_np(z):
    return np.maximum(0, z)

def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

x = np.array([-2.0, 0.0, 1.5], dtype=np.float32)
# TODO: assert close to torch and tensorflow outputs

# --- Conversión a tensores de PyTorch y TensorFlow ---
x_torch = torch.tensor(x)
x_tf = tf.convert_to_tensor(x)

# --- Cálculos con PyTorch y TensorFlow ---
relu_torch = torch.relu(x_torch).numpy()
relu_tf = tf.nn.relu(x_tf).numpy()

sigmoid_torch = torch.sigmoid(x_torch).numpy()
sigmoid_tf = tf.nn.sigmoid(x_tf).numpy()

# --- Comprobaciones (Assertions) ---

# Usamos np.allclose para comparar arrays de punto flotante y evitar problemas de precisión decimal
assert np.allclose(relu_np(x), relu_torch), "ReLU NumPy no coincide con PyTorch"
assert np.allclose(relu_np(x), relu_tf), "ReLU NumPy no coincide con TensorFlow"

assert np.allclose(sigmoid_np(x), sigmoid_torch), "Sigmoid NumPy no coincide con PyTorch"
assert np.allclose(sigmoid_np(x), sigmoid_tf), "Sigmoid NumPy no coincide con TensorFlow"

print("¡Todo correcto! Los resultados de NumPy coinciden con PyTorch y TensorFlow.")
print(f"Resultado ReLU: {relu_np(x)}")
print(f"Resultado Sigmoid: {sigmoid_np(x)}")


¡Todo correcto! Los resultados de NumPy coinciden con PyTorch y TensorFlow.
Resultado ReLU: [0.  0.  1.5]
Resultado Sigmoid: [0.11920293 0.5        0.8175745 ]


## Actividad 2 — Misma arquitectura, dos APIs

Define una red `Linear(10, 3)` + `ReLU` + `Linear(3, 1)` en **PyTorch** (`nn.Sequential`) y la misma en **Keras** (`Sequential`).

*Hint:* set seeds (`torch.manual_seed`, `tf.random.set_seed`) and use explicit init if you want to compare weights.


In [2]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

torch.manual_seed(42)
tf.random.set_seed(42)

# TODO: build torch_model and keras_model with the architecture above
x_np = np.ones((4, 10), dtype=np.float32) 
x_torch = torch.tensor(x_np)
x_keras = tf.convert_to_tensor(x_np)

torch_model = nn.Sequential(
    nn.Linear(10, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

# TODO: run a forward pass on random input shape (batch=4, features=10)
with torch.no_grad():
    nn.init.ones_(torch_model[0].weight)
    nn.init.zeros_(torch_model[0].bias)
    nn.init.ones_(torch_model[2].weight)
    nn.init.zeros_(torch_model[2].bias)

# Modelo en Keras / TensorFlow
keras_model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(3, activation='relu', 
                 kernel_initializer='ones', 
                 bias_initializer='zeros'),
    layers.Dense(1, 
                 kernel_initializer='ones', 
                 bias_initializer='zeros')
])

output_torch = torch_model(x_torch).detach().numpy()
output_keras = keras_model(x_keras).numpy()


print("Salida PyTorch:\n", output_torch)
print("\nSalida Keras:\n", output_keras)

assert np.allclose(output_torch, output_keras), "¡Las salidas no coinciden!"
print("\n¡Éxito! Ambos modelos devuelven exactamente el mismo resultado.")

Salida PyTorch:
 [[30.]
 [30.]
 [30.]
 [30.]]

Salida Keras:
 [[30.]
 [30.]
 [30.]
 [30.]]

¡Éxito! Ambos modelos devuelven exactamente el mismo resultado.


## Actividad 3 — Entrenamiento en mini-batch (conceptual + código corto)

Escribe un bucle de **una época** que: (1) muestree un batch sintético `X, y` para regresión, (2) calcule `MSE`, (3) haga `backward` / `gradient` y un paso de optimizador.

Elige **solo uno** de los dos frameworks para el bucle completo; en el otro, documenta en un comentario qué API usarías (`loss.backward`, `tape.gradient`, etc.).


In [4]:
import torch.optim as optim

# TODO: one epoch, one framework; comment the other API
torch.manual_seed(42)

# 1. Definimos el modelo, la función de pérdida y el optimizador
model = nn.Linear(in_features=10, out_features=1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Datos sintéticos simulados (4 batches de tamaño 32, con 10 características cada uno)
num_batches = 4
batch_size = 32
features = 10

print("--- Iniciando Época de Entrenamiento (PyTorch) ---")

--- Iniciando Época de Entrenamiento (PyTorch) ---


In [5]:
for batch_idx in range(num_batches):
    # (1) Muestrear/generar un batch sintético X, y
    X_batch = torch.randn(batch_size, features)
    y_batch = torch.randn(batch_size, 1)
    
    # Resetear los gradientes acumulados del paso anterior
    optimizer.zero_grad()
    
    # Forward pass: calcular las predicciones
    predictions = model(X_batch)
    
    # (2) Calcule MSE (Mean Squared Error)
    loss = criterion(predictions, y_batch)
    
    # (3) Hacer backward (cálculo de gradientes)
    loss.backward()
    
    # Paso del optimizador (actualizar pesos basándose en los gradientes)
    optimizer.step()
    
    print(f"Batch {batch_idx + 1}/{num_batches} - Pérdida (MSE): {loss.item():.4f}")

Batch 1/4 - Pérdida (MSE): 0.9897
Batch 2/4 - Pérdida (MSE): 1.4874
Batch 3/4 - Pérdida (MSE): 1.2274
Batch 4/4 - Pérdida (MSE): 0.8999


En PyTorch el enfoque es totalmente imperativo y directo, ya que los gradientes se acumulan de forma automática dentro de los propios parámetros del modelo en cuanto ejecutas loss.backward(). Por ello, el flujo se controla sencillamente limpiando los residuos previos con optimizer.zero_grad() y actualizando los pesos con optimizer.step().

Por el contrario, TensorFlow utiliza un gestor de contexto mediante with tf.GradientTape() as tape:, obligándote a envolver el cálculo de la pérdida y el pase hacia adelante para registrar las operaciones en una cinta virtual. Posteriormente, debes extraer explícitamente esos gradientes invocando a tape.gradient() y aplicarlos de forma manual a las variables entrenables a través de optimizer.apply_gradients().

En pocas palabras, PyTorch gestiona los gradientes internamente en el grafo de la pérdida, mientras que TensorFlow requiere que los captures y asocies de forma externa usando una cinta grabadora.